In [1]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


# # Dataset Generator for Composite DNA - Eta-Based Variable Ratio 2-Mix
# ## Cross-Platform Robustness Study: Nanopore (R21, B22, NP22, NPF22) + Newer Illumina (BOS22)
# ## Each profile uses its standard sequence length from the corresponding original dataset

# =============================================================================
# CELL 1: IMPORTS
# =============================================================================
import random
import numpy as np
import pickle
import json
import os
from collections import Counter
from datetime import datetime
import time


# =============================================================================
# CELL 2: CONFIGURATION
# =============================================================================

# ------------------- SELECT ERROR MODEL -------------------
# NEW PLATFORM OPTIONS:
#   "R21"    -> Oxford Nanopore MinION + Twist Bioscience (Rang et al. 2021)
#   "B22"    -> Nanopore MinION Short-read + Twist Bioscience (Bar-Lev et al. 2022)
#   "BOS22"  -> Illumina MiSeq 2022 + Twist Bioscience (very low error, newer Illumina)
#   "NP22"   -> Nanopore Pilot Nov-2022 + Twist Bioscience (highly non-uniform across bases)
#   "NPF22"  -> Nanopore Full Pool Nov-2022 + Twist Bioscience (comprehensive Nanopore)
#
# (Original profiles for reference, already have results: "erlich"/"EZ17", "grass"/"G15", "organick"/"O17")
# ----------------------------------------------------------
ERROR_MODEL = "R21"  # <-- CHANGE THIS TO SELECT ERROR MODEL

# ------------------- ETA-BASED ALPHABET PARAMETERS -------------------
ETA = 0.2  # Step size for mixture ratios
ELL_VALUES = [-2, -1, 0, 1, 2]  # Offset values: gives ratios 0.1/0.9, 0.3/0.7, 0.5/0.5, 0.7/0.3, 0.9/0.1

# Calculate vocab size: 4 pure + 6 pairs × len(ELL_VALUES)
NUM_PURE_BASES = 4
NUM_TWO_MIX_PAIRS = 6
VOCAB_SIZE = NUM_PURE_BASES + NUM_TWO_MIX_PAIRS * len(ELL_VALUES)  # 4 + 30 = 34

# All new profiles derive from the same Erlich/Twist oligo pool (72,000 oligos of 152nt,
# synthesized by Twist Bioscience), sequenced with different technologies.
# Standard oligo design: 152nt total, 16nt index/primer region → 136nt payload.
#
# Oligo Design Summary (standard lengths per original dataset):
#   Original profiles (different oligo pools / designs):
#     EZ17 : 152nt Twist, 16nt index  → seq_length = 136  (Erlich & Zielinski 2017)
#     G15  : 117nt CustomArray, 13nt index → seq_length = 104  (Grass et al. 2015)
#     O17  : 110nt Twist, 33nt index  → seq_length = 77   (Organick et al. 2018)
#
#   New profiles (same Erlich/Twist 152nt pool, different sequencing technologies):
#     R21  : 152nt Twist, 16nt index  → seq_length = 136  (Rang et al. 2021, Nanopore MinION)
#     B22  : 152nt Twist, 16nt index  → seq_length = 136  (Bar-Lev et al. 2022, MinION Short)
#     BOS22: 152nt Twist, 16nt index  → seq_length = 136  (Bar-Lev & Sabary 2022, Illumina MiSeq 2022)
#     NP22 : 152nt Twist, 16nt index  → seq_length = 136  (Bar-Lev & Sabary 2022, Nanopore Pilot)
#     NPF22: 152nt Twist, 16nt index  → seq_length = 136  (Bar-Lev & Sabary 2022, Nanopore Full)

ERROR_MODEL_SPECS = {
    # ---------- NEW NANOPORE PROFILES ----------
    "R21": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "R21",
        "platform": "Oxford Nanopore MinION",
        "synthesis": "Twist Bioscience",
        "reference": "Rang et al. 2021",
        "notes": "Standard MinION, high insertion rate (~1.65%)"
    },
    "B22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "B22",
        "platform": "Nanopore MinION Short",
        "synthesis": "Twist Bioscience",
        "reference": "Bar-Lev et al. 2022",
        "notes": "Short-read MinION, balanced errors (~1.1% each)"
    },
    "NP22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "NP22",
        "platform": "Nanopore Pilot Nov-2022",
        "synthesis": "Twist Bioscience",
        "reference": "Bar-Lev & Sabary et al. 2022",
        "notes": "Highly non-uniform per-base errors, C and T dominated"
    },
    "NPF22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "NPF22",
        "platform": "Nanopore Full Pool Nov-2022",
        "synthesis": "Twist Bioscience",
        "reference": "Bar-Lev & Sabary et al. 2022 (updated)",
        "notes": "Full pool Nanopore, high uniform errors (~1.5%)"
    },
    # ---------- NEW ILLUMINA PROFILE ----------
    "BOS22": {
        "full_length": 152,
        "index_length": 16,
        "seq_length": 136,  # 152 - 16 (same Erlich/Twist pool)
        "name": "BOS22",
        "platform": "Illumina MiSeq 2022",
        "synthesis": "Twist Bioscience",
        "reference": "Bar-Lev & Sabary et al. 2022",
        "notes": "Ultra-low error rates (~0.05%), newer Illumina generation"
    },
}

CONFIG = {
    # Error Model
    "error_model": ERROR_MODEL,
    "error_specs": ERROR_MODEL_SPECS[ERROR_MODEL],
    
    # Eta-Based Alphabet Parameters
    "eta": ETA,
    "ell_values": ELL_VALUES,
    "alphabet_mode": f"eta{ETA}_ell{len(ELL_VALUES)}",
    
    # Dataset Parameters
    "num_samples": 100000,
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    "max_coverage": 50,
    
    # Vocabulary
    "vocab_size": VOCAB_SIZE,
    
    # Output Directory
    "dataset_dir": "./dataset_cross_platform",
    
    # Reproducibility
    "seed": 42
}

# Create dataset name including error model
dataset_name = f"dna_{CONFIG['error_specs']['name']}_eta{ETA}"
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{dataset_name}_"
                          f"{CONFIG['num_samples']}_{CONFIG['max_coverage']}.pkl")

os.makedirs(CONFIG['dataset_dir'], exist_ok=True)

print(f"{'='*70}")
print(f"📋 CROSS-PLATFORM ETA-BASED DATASET GENERATION CONFIGURATION")
print(f"{'='*70}")
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_specs']['name']})")
print(f"   Platform: {CONFIG['error_specs']['platform']}")
print(f"   Synthesis: {CONFIG['error_specs']['synthesis']}")
print(f"   Reference: {CONFIG['error_specs']['reference']}")
print(f"   Notes: {CONFIG['error_specs']['notes']}")
print(f"   Oligo Design: {CONFIG['error_specs']['full_length']}nt total, "
      f"{CONFIG['error_specs']['index_length']}nt index → "
      f"{CONFIG['error_specs']['seq_length']}nt payload (standard)")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Eta: {ETA}, Ell Values: {ELL_VALUES}")
print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
print(f"   Theoretical Capacity: {np.log2(CONFIG['vocab_size']):.4f} bits/position")
print(f"   Num Samples: {CONFIG['num_samples']:,}")
print(f"   Max Coverage: {CONFIG['max_coverage']}")
print(f"   Output Path: {CONFIG['dataset_path']}")
print(f"{'='*70}")


# =============================================================================
# CELL 3: SEED FOR REPRODUCIBILITY
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: dataset_generator_eta_based-Erlich.py → Cell 3
# Copy: set_seed() function
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")


# =============================================================================
# CELL 4: ETA-BASED COMPOSITE DNA ALPHABET DEFINITIONS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: dataset_generator_eta_based-Erlich.py → Cell 4
# Copy: build_eta_based_alphabet(), print_alphabet_info()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def build_eta_based_alphabet(eta, ell_values):
    """
    Build the eta-based composite alphabet with variable mixture ratios.
    
    For each two-nucleotide pair (b1, b2), creates symbols with ratios:
        (0.5 + ℓ*η, 0.5 - ℓ*η) for each ℓ in ell_values
    
    Args:
        eta: Step size for mixture ratios (e.g., 0.2)
        ell_values: List of offset values (e.g., [-2, -1, 0, 1, 2])
    
    Returns:
        composite_map: Dict mapping symbol names to (nucleotide, probability) pairs
        symbol_to_idx: Dict mapping symbol names to class indices
        ideal_vectors: List of [A_prob, C_prob, G_prob, T_prob] vectors
    """
    
    # Pure bases: indices 0-3
    composite_map = {
        'A': [('A', 1.0)],
        'C': [('C', 1.0)],
        'G': [('G', 1.0)],
        'T': [('T', 1.0)],
    }
    
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
    ]
    
    # Define the 6 two-nucleotide pairs and their vector positions
    two_mix_pairs = [
        ('B1', 'A', 'C', 0, 1),  # A|C
        ('B2', 'A', 'G', 0, 2),  # A|G
        ('B3', 'A', 'T', 0, 3),  # A|T
        ('B4', 'C', 'G', 1, 2),  # C|G
        ('B5', 'C', 'T', 1, 3),  # C|T
        ('B6', 'G', 'T', 2, 3),  # G|T
    ]
    
    current_idx = 4  # Start after pure bases
    
    for pair_name, base1, base2, idx1, idx2 in two_mix_pairs:
        for ell in ell_values:
            prob1 = max(0.0, min(1.0, 0.5 + ell * eta))
            prob2 = max(0.0, min(1.0, 0.5 - ell * eta))
            
            if ell >= 0:
                symbol_name = f"{pair_name}_ell{ell}"
            else:
                symbol_name = f"{pair_name}_ell_neg{abs(ell)}"
            
            composite_map[symbol_name] = [(base1, prob1), (base2, prob2)]
            symbol_to_idx[symbol_name] = current_idx
            
            vec = [0.0, 0.0, 0.0, 0.0]
            vec[idx1] = prob1
            vec[idx2] = prob2
            ideal_vectors.append(vec)
            
            current_idx += 1
    
    return composite_map, symbol_to_idx, ideal_vectors


def print_alphabet_info(composite_map, symbol_to_idx, ideal_vectors):
    """Print detailed information about the alphabet."""
    print(f"\n🧬 Eta-Based Composite Alphabet (η={CONFIG['eta']}, ℓ∈{CONFIG['ell_values']}):")
    print(f"   Total Symbols: {len(symbol_to_idx)}")
    print(f"   Theoretical Capacity: {np.log2(len(symbol_to_idx)):.4f} bits/position")
    print(f"\n   {'Symbol':<16} {'Index':<6} {'Composition':<20} {'Ideal Vector [A,C,G,T]'}")
    print(f"   {'-'*70}")
    
    for sym, idx in sorted(symbol_to_idx.items(), key=lambda x: x[1]):
        comp_list = composite_map[sym]
        if len(comp_list) == 1:
            comp_str = comp_list[0][0]
        else:
            comp_str = f"{comp_list[0][0]}({comp_list[0][1]:.1f})|{comp_list[1][0]}({comp_list[1][1]:.1f})"
        vec = ideal_vectors[idx]
        vec_str = f"[{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}, {vec[3]:.2f}]"
        print(f"   {sym:<16} {idx:<6} {comp_str:<20} {vec_str}")


# Build the alphabet
COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS = build_eta_based_alphabet(CONFIG['eta'], CONFIG['ell_values'])
ALL_SYMBOLS = list(COMPOSITE_MAP.keys())
print_alphabet_info(COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS)


# =============================================================================
# CELL 5: ERROR RATES CLASS - CROSS-PLATFORM VERSION
# =============================================================================
# This extends the original ErrorRates class with 5 new profiles from
# the other_github_dataset repository, covering Nanopore and newer Illumina.

class ErrorRates:
    """Error rate configuration for multiple DNA sequencing technologies.
    
    All profiles use their standard oligo-derived sequence lengths.
    
    Original profiles (already evaluated, different oligo designs):
        EZ17  - Illumina MiSeq + Twist         (Erlich & Zielinski 2017)    152nt → n=136
        G15   - Illumina MiSeq + CustomArray    (Grass et al. 2015)         117nt → n=104
        O17   - Illumina NextSeq + Twist        (Organick et al. 2018)      110nt → n=77
    
    NEW cross-platform profiles (same 152nt Erlich/Twist pool, different sequencing):
        R21   - Oxford Nanopore MinION + Twist         (Rang et al. 2021)           152nt → n=136
        B22   - Nanopore MinION Short + Twist          (Bar-Lev et al. 2022)        152nt → n=136
        BOS22 - Illumina MiSeq 2022 + Twist            (Bar-Lev & Sabary 2022)     152nt → n=136
        NP22  - Nanopore Pilot Nov-2022 + Twist        (Bar-Lev & Sabary 2022)      152nt → n=136
        NPF22 - Nanopore Full Pool Nov-2022 + Twist    (Bar-Lev & Sabary 2022)      152nt → n=136
    """
    
    def __init__(self):
        self.general_errors = {'d': 0.0, 'ld': 0.0, 'i': 0.0, 's': 0.0}
        self.per_base_errors = {
            'A': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'C': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'G': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'T': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0}
        }
    
    # =====================================================================
    # ORIGINAL PROFILES (kept for reference / combined runs)
    # =====================================================================
    
    def set_EZ17_values(self):
        """Erlich & Zielinski 2017 (Illumina MiSeq + Twist) error profile. 152nt → n=136"""
        print("   >> Loading Erlich (EZ17) Error Profile...")
        self.general_errors = {'s': 1.32e-03, 'i': 5.81e-04, 'd': 9.58e-04, 'ld': 2.33e-04}
        self.per_base_errors['A'] = {'s': 0.00135, 'i': 0.00057, 'pi': 0.00059, 'd': 0.00099, 'ld': 0.00024}
        self.per_base_errors['C'] = {'s': 0.00135, 'i': 0.00059, 'pi': 0.00058, 'd': 0.00098, 'ld': 0.00023}
        self.per_base_errors['G'] = {'s': 0.00126, 'i': 0.00059, 'pi': 0.00057, 'd': 0.00094, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00132, 'i': 0.00058, 'pi': 0.00058, 'd': 0.00096, 'ld': 0.00023}
    
    def set_G15_values(self):
        """Grass et al. 2015 (Illumina MiSeq + CustomArray) error profile. 117nt → n=104"""
        print("   >> Loading Grass (G15) Error Profile...")
        self.general_errors = {'s': 5.84e-03, 'i': 8.57e-04, 'd': 5.37e-03, 'ld': 3.48e-04}
        self.per_base_errors['A'] = {'s': 0.00605, 'i': 0.0009,  'pi': 0.00092, 'd': 0.00543, 'ld': 0.00036}
        self.per_base_errors['C'] = {'s': 0.00563, 'i': 0.00083, 'pi': 0.00081, 'd': 0.00513, 'ld': 0.00034}
        self.per_base_errors['G'] = {'s': 0.00577, 'i': 0.00085, 'pi': 0.00087, 'd': 0.00539, 'ld': 0.00034}
        self.per_base_errors['T'] = {'s': 0.00591, 'i': 0.00084, 'pi': 0.00084, 'd': 0.00559, 'ld': 0.00036}
    
    def set_O17_values(self):
        """Organick et al. 2018 (Illumina NextSeq + Twist) error profile. 110nt → n=77"""
        print("   >> Loading Organick (O17) Error Profile...")
        self.general_errors = {'s': 2.52e-03, 'i': 4.14e-04, 'd': 6.94e-04, 'ld': 2.11e-04}
        self.per_base_errors['A'] = {'s': 0.00717, 'i': 0.0003,  'pi': 0.0012,  'd': 0.00201, 'ld': 0.00054}
        self.per_base_errors['C'] = {'s': 0.00034, 'i': 0.00007, 'pi': 0.00007, 'd': 0.00006, 'ld': 0.00001}
        self.per_base_errors['G'] = {'s': 0.00196, 'i': 0.00125, 'pi': 0.00029, 'd': 0.00058, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00055, 'i': 0.00006, 'pi': 0.00009, 'd': 0.00014, 'ld': 0.00006}
    
    # =====================================================================
    # NEW NANOPORE PROFILES
    # =====================================================================
    
    def set_R21_values(self):
        """Rang et al. 2021: Oxford Nanopore MinION + Twist Bioscience. 152nt → n=136
        Insertion-dominated: ~1.65% mean insertion rate. ~10x higher than EZ17."""
        print("   >> Loading R21 (Nanopore MinION + Twist) Error Profile...")
        self.general_errors = {'s': 1.08e-02, 'i': 1.65e-02, 'd': 1.18e-02, 'ld': 3.36e-03}
        self.per_base_errors['A'] = {'s': 1.039e-02, 'i': 1.61e-02,  'pi': 1.585e-02, 'd': 1.192e-02, 'ld': 0.314e-02}
        self.per_base_errors['C'] = {'s': 1.042e-02, 'i': 1.639e-02, 'pi': 1.578e-02, 'd': 1.26e-02,  'ld': 0.334e-02}
        self.per_base_errors['G'] = {'s': 1.094e-02, 'i': 1.604e-02, 'pi': 1.7e-02,   'd': 1.267e-02, 'ld': 0.337e-02}
        self.per_base_errors['T'] = {'s': 1.131e-02, 'i': 1.738e-02, 'pi': 1.729e-02, 'd': 1.327e-02, 'ld': 0.357e-02}
    
    def set_B22_values(self):
        """Bar-Lev et al. 2022: Nanopore MinION Short-read + Twist. 152nt → n=136
        Balanced: sub ~1.12%, ins ~1.08%, del ~0.79%."""
        print("   >> Loading B22 (Nanopore MinION Short + Twist) Error Profile...")
        self.general_errors = {'s': 1.12e-02, 'i': 1.08e-02, 'd': 7.87e-03, 'ld': 2.05e-03}
        self.per_base_errors['A'] = {'s': 0.01146, 'i': 0.01104, 'pi': 0.01096, 'd': 0.00798, 'ld': 0.0021}
        self.per_base_errors['C'] = {'s': 0.01117, 'i': 0.01092, 'pi': 0.01088, 'd': 0.00804, 'ld': 0.00208}
        self.per_base_errors['G'] = {'s': 0.01129, 'i': 0.01073, 'pi': 0.01071, 'd': 0.00792, 'ld': 0.00201}
        self.per_base_errors['T'] = {'s': 0.01094, 'i': 0.01039, 'pi': 0.01053, 'd': 0.00769, 'ld': 0.00201}
    
    def set_NP22_values(self):
        """Nanopore Pilot Pool Nov-2022 + Twist. 152nt → n=136
        HIGHLY non-uniform: T has ~5x higher substitution than A."""
        print("   >> Loading NP22 (Nanopore Pilot Nov-2022 + Twist) Error Profile...")
        self.general_errors = {'s': 1.29e-02, 'i': 1.16e-02, 'd': 9.75e-03, 'ld': 2.82e-03}
        self.per_base_errors['A'] = {'s': 0.52e-2,  'i': 0.644e-2, 'pi': 0.582e-2, 'd': 0.373e-2, 'ld': 0.098e-2}
        self.per_base_errors['C'] = {'s': 1.498e-2, 'i': 1.331e-2, 'pi': 1.34e-2,  'd': 1.178e-2, 'ld': 0.336e-2}
        self.per_base_errors['G'] = {'s': 0.638e-2, 'i': 0.735e-2, 'pi': 0.667e-2, 'd': 0.474e-2, 'ld': 0.128e-2}
        self.per_base_errors['T'] = {'s': 2.437e-2, 'i': 1.867e-2, 'pi': 1.983e-2, 'd': 1.903e-2, 'ld': 0.55e-2}
    
    def set_NPF22_values(self):
        """Nanopore Full Pool Nov-2022 (updated) + Twist. 152nt → n=136
        Highest overall errors: sub ~1.56%, ins ~1.24%, del ~0.98%."""
        print("   >> Loading NPF22 (Nanopore Full Pool Nov-2022 + Twist) Error Profile...")
        self.general_errors = {'s': 1.56e-02, 'i': 1.24e-02, 'd': 9.79e-03, 'ld': 2.67e-03}
        self.per_base_errors['A'] = {'s': 1.7e-2,   'i': 1.32e-2,  'pi': 1.343e-2, 'd': 1.095e-2, 'ld': 0.294e-2}
        self.per_base_errors['C'] = {'s': 1.606e-2, 'i': 1.271e-2, 'pi': 1.272e-2, 'd': 1.038e-2, 'ld': 0.277e-2}
        self.per_base_errors['G'] = {'s': 1.593e-2, 'i': 1.256e-2, 'pi': 1.255e-2, 'd': 1.014e-2, 'ld': 0.273e-2}
        self.per_base_errors['T'] = {'s': 1.358e-2, 'i': 1.116e-2, 'pi': 1.094e-2, 'd': 0.849e-2, 'ld': 0.224e-2}
    
    # =====================================================================
    # NEW ILLUMINA PROFILE
    # =====================================================================
    
    def set_BOS22_values(self):
        """Illumina MiSeq 2022 + Twist (second pilot, 08-09-2022). 152nt → n=136
        Ultra-low: ~10x lower than EZ17. T dominates substitution."""
        print("   >> Loading BOS22 (Illumina MiSeq 2022 + Twist) Error Profile...")
        self.general_errors = {'s': 5.29e-04, 'i': 5.42e-05, 'd': 7.08e-05, 'ld': 1.81e-05}
        self.per_base_errors['A'] = {'s': 0.004e-2,  'i': 0.004e-2, 'pi': 0.0,     'd': 0.0,     'ld': 0.0}
        self.per_base_errors['C'] = {'s': 0.024e-2,  'i': 0.006e-2, 'pi': 0.002e-2, 'd': 0.003e-2, 'ld': 0.001e-2}
        self.per_base_errors['G'] = {'s': 0.004e-2,  'i': 0.005e-2, 'pi': 0.0,     'd': 0.0,     'ld': 0.0}
        self.per_base_errors['T'] = {'s': 0.175e-2,  'i': 0.007e-2, 'pi': 0.018e-2, 'd': 0.024e-2, 'ld': 0.006e-2}
    
    # =====================================================================
    # DISPATCHER
    # =====================================================================
    
    def set_values_by_model(self, model_name):
        """Set error values based on model name string."""
        dispatch = {
            "erlich": self.set_EZ17_values, "EZ17": self.set_EZ17_values,
            "grass": self.set_G15_values,   "G15":  self.set_G15_values,
            "organick": self.set_O17_values,"O17":  self.set_O17_values,
            "R21": self.set_R21_values, "B22": self.set_B22_values,
            "BOS22": self.set_BOS22_values, "NP22": self.set_NP22_values,
            "NPF22": self.set_NPF22_values,
        }
        if model_name not in dispatch:
            raise ValueError(f"Unknown error model: '{model_name}'. Available: {list(dispatch.keys())}")
        dispatch[model_name]()
    
    def print_current_values(self):
        print("\n   --- Error Configuration ---")
        print(f"   General: sub={self.general_errors['s']:.5f}, "
              f"ins={self.general_errors['i']:.5f}, "
              f"del={self.general_errors['d']:.5f}, "
              f"ldel={self.general_errors['ld']:.5f}")
        total_general = sum(self.general_errors.values())
        print(f"   Total error rate: {total_general:.5f} ({total_general*100:.3f}%)")
        print(f"   {'Base':<6} {'Sub':>10} {'Ins':>10} {'Del':>10} {'LDel':>10}")
        print(f"   {'-'*46}")
        for base in ['A', 'C', 'G', 'T']:
            r = self.per_base_errors[base]
            print(f"   {base:<6} {r['s']:>10.5f} {r['i']:>10.5f} {r['d']:>10.5f} {r['ld']:>10.5f}")
        print(f"   {'-'*46}")
    
    def get_error_summary(self):
        return {
            'general': dict(self.general_errors),
            'per_base': {b: dict(v) for b, v in self.per_base_errors.items()},
            'total_rate': sum(self.general_errors.values())
        }


# =============================================================================
# CELL 6: SEQUENCE GENERATION FUNCTIONS - ETA-BASED
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: dataset_generator_eta_based-Erlich.py → Cell 6
# Copy: generate_composite_sequence(), realize_sequence_eta(), apply_ids_noise()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def generate_composite_sequence(length, composite_map):
    """Generates a random sequence of composite symbols."""
    symbols = list(composite_map.keys())
    return [random.choice(symbols) for _ in range(length)]


def realize_sequence_eta(composite_seq, composite_map):
    """
    Converts composite symbols to a single DNA realization.
    For eta-based alphabet, each symbol has specific probabilities for its nucleotides.
    """
    realized = []
    for sym in composite_seq:
        components = composite_map[sym]
        if len(components) == 1:
            nucleotide = components[0][0]
        else:
            bases = [comp[0] for comp in components]
            probs = [comp[1] for comp in components]
            nucleotide = random.choices(bases, weights=probs, k=1)[0]
        realized.append(nucleotide)
    return "".join(realized)


def apply_ids_noise(sequence, error_profile):
    """Apply Insertion, Deletion, Substitution noise to a DNA sequence."""
    bases = ['A', 'C', 'G', 'T']
    noisy_seq = []
    
    for base in sequence:
        if base not in bases:
            continue
        rates = error_profile.per_base_errors[base]
        p_sub = rates['s']
        p_ins = rates['i']
        p_del = rates['d']
        
        if random.random() < p_del:
            continue
        if random.random() < p_ins:
            noisy_seq.append(random.choice(bases))
        if random.random() < p_sub:
            options = [b for b in bases if b != base]
            noisy_seq.append(random.choice(options))
        else:
            noisy_seq.append(base)
    return "".join(noisy_seq)


# =============================================================================
# CELL 7: MAIN DATASET GENERATION FUNCTION
# =============================================================================

def generate_dataset_eta(config, composite_map, symbol_to_idx, ideal_vectors):
    """Generate dataset with eta-based composite alphabet for any error model."""
    errors = ErrorRates()
    errors.set_values_by_model(config['error_model'])
    errors.print_current_values()
    
    num_samples = config['num_samples']
    seq_length = config['seq_length']
    coverage = config['max_coverage']
    filename = config['dataset_path']
    error_specs = config['error_specs']
    
    dataset = {
        'metadata': {
            'type': f'Composite DNA Eta-Based (η={config["eta"]})',
            'error_profile': f'{error_specs["name"]} ({error_specs["platform"]})',
            'error_model': config['error_model'],
            'platform': error_specs['platform'],
            'synthesis': error_specs['synthesis'],
            'reference': error_specs['reference'],
            'full_length': error_specs['full_length'],
            'index_length': error_specs['index_length'],
            'error_summary': errors.get_error_summary(),
            'num_samples': num_samples,
            'seq_length': seq_length,
            'coverage_depth': coverage,
            'vocab_size': config['vocab_size'],
            'eta': config['eta'],
            'ell_values': config['ell_values'],
            'alphabet_mode': config['alphabet_mode'],
            'symbols': list(symbol_to_idx.keys()),
            'symbol_to_idx': symbol_to_idx,
            'ideal_vectors': ideal_vectors,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'seed': config['seed']
        },
        'data': []
    }
    
    print(f"\n{'='*60}")
    print(f"🔄 GENERATING DATASET")
    print(f"{'='*60}")
    print(f"   Error Model: {config['error_model']} ({error_specs['platform']})")
    print(f"   Oligo: {error_specs['full_length']}nt − {error_specs['index_length']}nt index = {seq_length}nt payload (standard)")
    print(f"   Samples: {num_samples:,}")
    print(f"   Sequence Length: {seq_length}")
    print(f"   Coverage Depth: {coverage}")
    print(f"   Alphabet: η={config['eta']}, {config['vocab_size']} classes")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    for i in range(num_samples):
        clean_composite_seq = generate_composite_sequence(seq_length, composite_map)
        cluster_reads = []
        for _ in range(coverage):
            realized_dna = realize_sequence_eta(clean_composite_seq, composite_map)
            noisy_read = apply_ids_noise(realized_dna, errors)
            cluster_reads.append(noisy_read)
        
        sample = {'id': i, 'label': clean_composite_seq, 'cluster': cluster_reads}
        dataset['data'].append(sample)
        
        if (i + 1) % 10000 == 0:
            elapsed = time.time() - start_time
            samples_per_sec = (i + 1) / elapsed
            eta_time = (num_samples - i - 1) / samples_per_sec
            print(f"   Processed {i+1:,}/{num_samples:,} | "
                  f"Speed: {samples_per_sec:.1f} samples/s | "
                  f"ETA: {eta_time:.1f}s")

    with open(filename, 'wb') as f:
        pickle.dump(dataset, f)
    
    total_time = time.time() - start_time
    
    print(f"\n{'='*60}")
    print(f"✅ DATASET GENERATION COMPLETE")
    print(f"{'='*60}")
    print(f"   Error Model: {config['error_model']} ({error_specs['platform']})")
    print(f"   Output File: {filename}")
    print(f"   Total Time: {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"   File Size: {os.path.getsize(filename) / (1024*1024):.1f} MB")
    
    print(f"\n🔍 Sample 0:")
    print(f"   Label (first 10): {dataset['data'][0]['label'][:10]}")
    print(f"   Read 1 (first 30): {dataset['data'][0]['cluster'][0][:30]}...")
    
    return dataset


# =============================================================================
# CELL 8: ANALYZE DATASET STATISTICS
# =============================================================================
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# SAME AS: dataset_generator_eta_based-Erlich.py → Cell 8
# Copy: analyze_dataset_eta() function
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def analyze_dataset_eta(dataset, symbol_to_idx, config):
    """Analyze and print dataset statistics for eta-based alphabet."""
    print(f"\n{'='*60}")
    print(f"📊 DATASET STATISTICS")
    print(f"{'='*60}")
    
    all_symbols = []
    for sample in dataset['data']:
        all_symbols.extend(sample['label'])
    
    counter = Counter(all_symbols)
    total_symbols = len(all_symbols)
    
    pure_bases = ['A', 'C', 'G', 'T']
    pure_count = sum(counter.get(s, 0) for s in pure_bases)
    mix_count = total_symbols - pure_count
    
    print(f"\n📈 Symbol Type Distribution:")
    print(f"   Pure Bases (A,C,G,T): {pure_count:,} ({100*pure_count/total_symbols:.1f}%)")
    print(f"   Eta-Mix Symbols:      {mix_count:,} ({100*mix_count/total_symbols:.1f}%)")
    
    all_read_lengths = []
    for sample in dataset['data']:
        for read in sample['cluster']:
            all_read_lengths.append(len(read))
    
    print(f"\n📏 Read Length Statistics:")
    print(f"   Original Length: {dataset['metadata']['seq_length']}")
    print(f"   Mean Read Length: {np.mean(all_read_lengths):.2f}")
    print(f"   Std Read Length: {np.std(all_read_lengths):.2f}")
    print(f"   Min Read Length: {np.min(all_read_lengths)}")
    print(f"   Max Read Length: {np.max(all_read_lengths)}")


# =============================================================================
# CELL 9: ERROR PROFILE COMPARISON UTILITY
# =============================================================================

def print_all_profiles_comparison():
    """Print a comparison table of all available error profiles with oligo design details."""
    all_specs = {
        "EZ17":  {"platform": "Illumina MiSeq + Twist",         "full": 152, "idx": 16, "n": 136},
        "G15":   {"platform": "Illumina MiSeq + CustomArray",    "full": 117, "idx": 13, "n": 104},
        "O17":   {"platform": "Illumina NextSeq + Twist",        "full": 110, "idx": 33, "n": 77},
        "R21":   {"platform": "Nanopore MinION + Twist",         "full": 152, "idx": 16, "n": 136},
        "B22":   {"platform": "Nanopore MinION Short + Twist",   "full": 152, "idx": 16, "n": 136},
        "BOS22": {"platform": "Illumina MiSeq 2022 + Twist",     "full": 152, "idx": 16, "n": 136},
        "NP22":  {"platform": "Nanopore Pilot 2022 + Twist",     "full": 152, "idx": 16, "n": 136},
        "NPF22": {"platform": "Nanopore Full 2022 + Twist",      "full": 152, "idx": 16, "n": 136},
    }
    
    print(f"\n{'='*115}")
    print(f"📊 CROSS-PLATFORM ERROR PROFILE COMPARISON (Eta-Based, Standard Sequence Lengths)")
    print(f"{'='*115}")
    print(f"\n{'Profile':<8} {'Platform':<32} {'Oligo':>5} {'Idx':>4} {'n':>4} "
          f"{'Sub(%)':>8} {'Ins(%)':>8} {'Del(%)':>8} {'LDel(%)':>8} {'Total(%)':>9}")
    print(f"{'-'*115}")
    
    for name in ["EZ17", "G15", "O17", "R21", "B22", "BOS22", "NP22", "NPF22"]:
        err = ErrorRates()
        err.set_values_by_model(name)
        g = err.general_errors
        total = sum(g.values())
        sp = all_specs[name]
        print(f"{name:<8} {sp['platform']:<32} {sp['full']:>5} {sp['idx']:>4} {sp['n']:>4} "
              f"{g['s']*100:>8.4f} {g['i']*100:>8.4f} "
              f"{g['d']*100:>8.4f} {g['ld']*100:>8.4f} "
              f"{total*100:>9.4f}")
    
    print(f"\n   Illumina profiles: EZ17 (n=136), G15 (n=104), O17 (n=77), BOS22 (n=136)")
    print(f"   Nanopore profiles: R21 (n=136), B22 (n=136), NP22 (n=136), NPF22 (n=136)")
    print(f"   Note: New profiles share n=136 (same Erlich/Twist 152nt oligo pool)")
    print(f"{'='*115}")

print_all_profiles_comparison()







📋 CROSS-PLATFORM ETA-BASED DATASET GENERATION CONFIGURATION
   Error Model: R21 (R21)
   Platform: Oxford Nanopore MinION
   Synthesis: Twist Bioscience
   Reference: Rang et al. 2021
   Notes: Standard MinION, high insertion rate (~1.65%)
   Oligo Design: 152nt total, 16nt index → 136nt payload (standard)
   Sequence Length: 136
   Eta: 0.2, Ell Values: [-2, -1, 0, 1, 2]
   Vocab Size: 34 classes
   Theoretical Capacity: 5.0875 bits/position
   Num Samples: 100,000
   Max Coverage: 50
   Output Path: ./dataset_cross_platform/dna_R21_eta0.2_100000_50.pkl
🎲 Random seed set to: 42

🧬 Eta-Based Composite Alphabet (η=0.2, ℓ∈[-2, -1, 0, 1, 2]):
   Total Symbols: 34
   Theoretical Capacity: 5.0875 bits/position

   Symbol           Index  Composition          Ideal Vector [A,C,G,T]
   ----------------------------------------------------------------------
   A                0      A                    [1.00, 0.00, 0.00, 0.00]
   C                1      C                    [0.00, 1.00, 0.00,

In [2]:
# print("\n" + "="*70)
# print(f"🧬 COMPOSITE DNA DATASET GENERATOR - ETA-BASED CROSS-PLATFORM")
# print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_specs']['name']})")
# print(f"   Platform: {CONFIG['error_specs']['platform']}")
# print(f"   Oligo: {CONFIG['error_specs']['full_length']}nt − {CONFIG['error_specs']['index_length']}nt = "
#       f"{CONFIG['error_specs']['seq_length']}nt (standard)")
# print(f"   Eta: {CONFIG['eta']}, Vocab Size: {CONFIG['vocab_size']} classes")
# print("="*70)

# if os.path.exists(CONFIG['dataset_path']):
#     print(f"\n⚠️  Dataset already exists: {CONFIG['dataset_path']}")
#     response = input("   Overwrite? (y/n): ").strip().lower()
#     if response != 'y':
#         print("   Aborted.")
#         exit()

# dataset = generate_dataset_eta(CONFIG, COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS)
# analyze_dataset_eta(dataset, SYMBOL_TO_IDX, CONFIG)

# print(f"\n{'='*70}")
# print(f"🎉 Dataset generation completed successfully!")
# print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_specs']['platform']})")
# print(f"   Eta: {CONFIG['eta']}, Classes: {CONFIG['vocab_size']}")
# print(f"   File: {CONFIG['dataset_path']}")
# print(f"{'='*70}")

In [3]:
# =============================================================================
# CELL 11: BATCH GENERATION HELPER (Optional)
# =============================================================================
# Uncomment below to generate datasets for ALL new profiles in one go.


BATCH_PROFILES = ["R21", "B22", "BOS22", "NP22", "NPF22"]

for profile in BATCH_PROFILES:
    print(f"\n{'#'*70}")
    print(f"# GENERATING: {profile} × eta{ETA}")
    print(f"{'#'*70}")
    
    batch_config = CONFIG.copy()
    batch_config['error_model'] = profile
    batch_config['error_specs'] = ERROR_MODEL_SPECS[profile]
    batch_config['seq_length'] = ERROR_MODEL_SPECS[profile]['seq_length']
    
    dataset_name = f"dna_{profile}_eta{ETA}"
    batch_config['dataset_path'] = (
        f"{batch_config['dataset_dir']}/{dataset_name}_"
        f"{batch_config['num_samples']}_{batch_config['max_coverage']}.pkl"
    )
    
    set_seed(batch_config['seed'])
    ds = generate_dataset_eta(batch_config, COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS)
    analyze_dataset_eta(ds, SYMBOL_TO_IDX, batch_config)
    print(f"✅ Done: {profile}")



######################################################################
# GENERATING: R21 × eta0.2
######################################################################
   >> Loading R21 (Nanopore MinION + Twist) Error Profile...

   --- Error Configuration ---
   General: sub=0.01080, ins=0.01650, del=0.01180, ldel=0.00336
   Total error rate: 0.04246 (4.246%)
   Base          Sub        Ins        Del       LDel
   ----------------------------------------------
   A         0.01039    0.01610    0.01192    0.00314
   C         0.01042    0.01639    0.01260    0.00334
   G         0.01094    0.01604    0.01267    0.00337
   T         0.01131    0.01738    0.01327    0.00357
   ----------------------------------------------

🔄 GENERATING DATASET
   Error Model: R21 (Oxford Nanopore MinION)
   Oligo: 152nt − 16nt index = 136nt payload (standard)
   Samples: 100,000
   Sequence Length: 136
   Coverage Depth: 50
   Alphabet: η=0.2, 34 classes
   Processed 10,000/100,000 | Speed: 68.5 sam

   Max Read Length: 138
✅ Done: BOS22

######################################################################
# GENERATING: NP22 × eta0.2
######################################################################
   >> Loading NP22 (Nanopore Pilot Nov-2022 + Twist) Error Profile...

   --- Error Configuration ---
   General: sub=0.01290, ins=0.01160, del=0.00975, ldel=0.00282
   Total error rate: 0.03707 (3.707%)
   Base          Sub        Ins        Del       LDel
   ----------------------------------------------
   A         0.00520    0.00644    0.00373    0.00098
   C         0.01498    0.01331    0.01178    0.00336
   G         0.00638    0.00735    0.00474    0.00128
   T         0.02437    0.01867    0.01903    0.00550
   ----------------------------------------------

🔄 GENERATING DATASET
   Error Model: NP22 (Nanopore Pilot Nov-2022)
   Oligo: 152nt − 16nt index = 136nt payload (standard)
   Samples: 100,000
   Sequence Length: 136
   Coverage Depth: 50
   Alphabet: η=0.2, 34 cla